In [ ]:
from pathlib import Path

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip intel-image-classification.zip

In [ ]:
import cv2

building = cv2.imread('seg_train/seg_train/buildings/10006.jpg')
building.shape

In [ ]:
from google.colab.patches import cv2_imshow

cv2_imshow(building)

In [ ]:
forest = cv2.imread('seg_train/seg_train/forest/10007.jpg')
cv2_imshow(forest)

In [ ]:
class_folder_paths = []
for child in Path('seg_train/seg_train').iterdir():
  class_folder_paths.append(child)

In [ ]:
class_folder_paths

In [ ]:
for folder_path in class_folder_paths[2:]:
  for child in folder_path.iterdir():
    print(child)
    break
  break

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_data_gen = ImageDataGenerator(horizontal_flip=True)

train_generator = train_data_gen.flow_from_directory('seg_train/seg_train',
                                                     target_size=(150, 150),
                                                     color_mode='rgb',
                                                     batch_size=32,
                                                     class_mode='categorical',
                                                     shuffle=True)


test_data_gen = ImageDataGenerator()

test_generator = test_data_gen.flow_from_directory('seg_test/seg_test',
                                                     target_size=(150, 150),
                                                     color_mode='rgb',
                                                     batch_size=32,
                                                    class_mode='categorical',
                                                     shuffle=False)

In [ ]:
labels=train_generator.class_indices
class_mapping = dict((v, k) for k, v in labels.items())
class_mapping

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Lambda, GlobalAveragePooling2D, Dropout, Dense


before_mobilenet = Sequential([
    Input((150, 150, 3)),
    Lambda(preprocess_input)
])

mobilenet = MobileNetV2(input_shape=(150, 150, 3), include_top=False)

after_mobilenet = Sequential([
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(units=6, activation='softmax')
])


model = Sequential([
    before_mobilenet,
    mobilenet,
    after_mobilenet
])




In [ ]:
from tensorflow.keras.optimizers import Adam

opt = Adam(0.00001)

model.compile(
    optimizer=opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.build((None, 150, 150, 3))
before_mobilenet.summary()
mobilenet.summary()
after_mobilenet.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

train_cb = ModelCheckpoint('model.keras',
                           save_best_only=True)
model.fit(train_generator,
          validation_data=test_generator,
          callbacks=[train_cb],
          epochs=7)

In [ ]:
!zip -r model.zip model.keras

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflfite_model

In [ ]:
tflite_model

In [ ]:
with open('model.tflite', 'wb') as f:
  f.write(tflite_model)